In [1]:
from __future__ import annotations

import os
from datetime import datetime, timezone
from getpass import getpass
from typing import Any

import httpx
import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, MetadataQuery

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("GOOGLE_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-Goog-Studio-Api-Key": os.environ["GOOGLE_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [21]:
GOOGLE_EMBEDDING_MODEL = "gemini-embedding-001"
GOOGLE_GENERATIVE_MODEL = "gemini-2.5-flash"

print("Embedding model:", GOOGLE_EMBEDDING_MODEL)
print("Generative model:", GOOGLE_GENERATIVE_MODEL)

Embedding model: gemini-embedding-001
Generative model: gemini-2.5-flash


In [5]:
def get_json(
    url: str,
    *,
    params: dict[str, Any] | None = None,
) -> dict[str, Any]:
    headers = {
        "User-Agent": "PythonPracticeNotebook/1.0 educational project",
        "Accept": "application/json",
    }

    with httpx.Client(
        timeout=30.0,
        follow_redirects=True,
        headers=headers,
    ) as http_client:
        response = http_client.get(url, params=params)
        response.raise_for_status()
        return response.json()

In [6]:
TECH_QUERIES = [
    "python",
    "fastapi",
    "postgres",
    "docker",
    "kubernetes",
    "vector database",
    "rag",
    "llm",
    "open source ai",
    "developer tools",
]

In [7]:
def fetch_hn_stories(
    query: str,
    *,
    hits_per_page: int = 10,
) -> list[dict[str, Any]]:
    url = "https://hn.algolia.com/api/v1/search_by_date"

    data = get_json(
        url,
        params={
            "query": query,
            "tags": "story",
            "hitsPerPage": hits_per_page,
        },
    )

    return data.get("hits", [])

In [8]:
def parse_datetime_to_rfc3339(value: str | None) -> str:
    if not value:
        return datetime.now(timezone.utc).isoformat()

    if value.endswith("Z"):
        value = value.replace("Z", "+00:00")

    return datetime.fromisoformat(value).astimezone(timezone.utc).isoformat()


def normalize_hn_story(
    story: dict[str, Any],
    *,
    query: str,
) -> dict[str, Any]:
    title = story.get("title") or story.get("story_title") or "Untitled"
    url = story.get("url") or ""
    author = story.get("author") or "unknown"

    points = story.get("points")
    points = int(points) if points is not None else 0

    comments_count = story.get("num_comments")
    comments_count = int(comments_count) if comments_count is not None else 0

    object_id = str(story.get("objectID") or story.get("story_id") or "")
    created_at = parse_datetime_to_rfc3339(story.get("created_at"))

    hn_url = f"https://news.ycombinator.com/item?id={object_id}"

    summary = (
        f"Hacker News story about {query}. "
        f"Title: {title}. "
        f"Author: {author}. "
        f"Points: {points}. "
        f"Comments: {comments_count}. "
        f"URL: {url if url else hn_url}."
    )

    return {
        "object_id": object_id,
        "query": query,
        "title": title,
        "url": url,
        "hn_url": hn_url,
        "author": author,
        "points": points,
        "comments_count": comments_count,
        "created_at": created_at,
        "summary": summary,
    }

In [9]:
raw_stories = []

for query in TECH_QUERIES:
    print("Fetching:", query)

    try:
        stories = fetch_hn_stories(query, hits_per_page=8)
        for story in stories:
            raw_stories.append(
                normalize_hn_story(
                    story,
                    query=query,
                )
            )
    except Exception as error:
        print("Failed:", query)
        print(repr(error))

unique_stories_by_id = {}

for story in raw_stories:
    object_id = story["object_id"]

    if not object_id:
        continue

    if object_id not in unique_stories_by_id:
        unique_stories_by_id[object_id] = story

tech_stories = list(unique_stories_by_id.values())

print("Raw stories:", len(raw_stories))
print("Unique stories:", len(tech_stories))

Fetching: python
Fetching: fastapi
Fetching: postgres
Fetching: docker
Fetching: kubernetes
Fetching: vector database
Fetching: rag
Fetching: llm
Fetching: open source ai
Fetching: developer tools
Raw stories: 80
Unique stories: 74


In [10]:
for story in tech_stories[:10]:
    print("Title:", story["title"])
    print("Query:", story["query"])
    print("Points:", story["points"])
    print("Comments:", story["comments_count"])
    print("URL:", story["url"] or story["hn_url"])
    print("Summary:", story["summary"])
    print("-" * 100)

Title: Python JIT project was asked to pause development
Query: python
Points: 2
Comments: 0
URL: https://discuss.python.org/t/an-announcement-from-the-steering-council-regarding-the-jit-project/107638
Summary: Hacker News story about python. Title: Python JIT project was asked to pause development. Author: kbumsik. Points: 2. Comments: 0. URL: https://discuss.python.org/t/an-announcement-from-the-steering-council-regarding-the-jit-project/107638.
----------------------------------------------------------------------------------------------------
Title: Running Python code in a sandbox with MicroPython and WASM
Query: python
Points: 7
Comments: 1
URL: https://simonwillison.net/2026/Jun/6/micropython-in-a-sandbox/
Summary: Hacker News story about python. Title: Running Python code in a sandbox with MicroPython and WASM. Author: theanonymousone. Points: 7. Comments: 1. URL: https://simonwillison.net/2026/Jun/6/micropython-in-a-sandbox/.
---------------------------------------------------

In [11]:
COLLECTION_NAME = "GoogleHNTechStory"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

hn_stories = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_google_gemini(
        name="story_vector",
        source_properties=[
            "query",
            "title",
            "summary",
        ],
        model=GOOGLE_EMBEDDING_MODEL,
    ),
    generative_config=wvc.config.Configure.Generative.google_gemini(
        model=GOOGLE_GENERATIVE_MODEL,
        temperature=0.2,
        max_output_tokens=900,
    ),
    properties=[
        wvc.config.Property(name="object_id", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="query", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="title", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="url", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="hn_url", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="author", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="points", data_type=wvc.config.DataType.INT),
        wvc.config.Property(name="comments_count", data_type=wvc.config.DataType.INT),
        wvc.config.Property(name="created_at", data_type=wvc.config.DataType.DATE),
        wvc.config.Property(name="summary", data_type=wvc.config.DataType.TEXT),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: GoogleHNTechStory


In [12]:
if not tech_stories:
    raise RuntimeError("No tech stories to import.")

result = hn_stories.data.insert_many(tech_stories)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [13]:
response = hn_stories.aggregate.over_all(total_count=True)

print("Total objects:", response.total_count)

Total objects: 74


In [14]:
response = hn_stories.query.fetch_objects(
    limit=10,
    return_properties=[
        "title",
        "query",
        "points",
        "comments_count",
        "created_at",
        "url",
        "hn_url",
    ],
)

for obj in response.objects:
    print("Title:", obj.properties["title"])
    print("Query:", obj.properties["query"])
    print("Points:", obj.properties["points"])
    print("Comments:", obj.properties["comments_count"])
    print("Date:", obj.properties["created_at"])
    print("URL:", obj.properties["url"] or obj.properties["hn_url"])
    print("-" * 100)

Title: Waiting times for healthcare in Europe: The worst countries ranked
Query: rag
Points: 1
Comments: 1
Date: 2026-06-06 15:04:42+00:00
URL: https://www.euronews.com/health/2026/06/03/waiting-times-for-healthcare-in-europe-the-worst-countries-ranked
----------------------------------------------------------------------------------------------------
Title: Migrating from Firestore to PostgreSQL
Query: postgres
Points: 2
Comments: 0
Date: 2026-06-05 17:42:35+00:00
URL: https://medium.com/@ValentinMouret/functionally-migrating-from-firestore-to-postgresql-64947b5dff0d
----------------------------------------------------------------------------------------------------
Title: Show HN: Microcodegen.py – PRD → FastAPI app, one file, no LLM calls
Query: fastapi
Points: 3
Comments: 0
Date: 2026-05-22 21:29:35+00:00
URL: https://github.com/Anioko/microcodegen
----------------------------------------------------------------------------------------------------
Title: Show HN: ARQ Dashboard – we

In [15]:
response = hn_stories.query.near_text(
    query="AI developer tools, vector databases, RAG applications and LLM infrastructure",
    target_vector="story_vector",
    limit=8,
    return_properties=[
        "title",
        "query",
        "points",
        "comments_count",
        "url",
        "hn_url",
        "summary",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Query:", obj.properties["query"])
    print("Points:", obj.properties["points"])
    print("Comments:", obj.properties["comments_count"])
    print("URL:", obj.properties["url"] or obj.properties["hn_url"])
    print("-" * 100)

Distance: 0.25231319665908813
Title: Show HN: SynapCores – AI-native database (vector, graph, SQL, AutoML, LLM)
Query: vector database
Points: 2
Comments: 0
URL: https://synapcores.com
----------------------------------------------------------------------------------------------------
Distance: 0.25846779346466064
Title: A Vector Lakebase is all you need for all AI workloads
Query: vector database
Points: 2
Comments: 0
URL: https://zilliz.com/blog/from-vector-database-to-vector-lakebase
----------------------------------------------------------------------------------------------------
Distance: 0.2599292993545532
Title: I Replaced My AI Agent's Flat Fact Store with a Graph Database
Query: vector database
Points: 10
Comments: 5
URL: https://news.ycombinator.com/item?id=48383578
----------------------------------------------------------------------------------------------------
Distance: 0.26461488008499146
Title: SQLiteGraph – embedded graph database with HNSW vector search
Query: vect

In [16]:
response = hn_stories.query.near_text(
    query="Python backend stack with FastAPI, PostgreSQL, Docker and Kubernetes",
    target_vector="story_vector",
    limit=8,
    return_properties=[
        "title",
        "query",
        "points",
        "comments_count",
        "url",
        "hn_url",
        "summary",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Query:", obj.properties["query"])
    print("Points:", obj.properties["points"])
    print("Comments:", obj.properties["comments_count"])
    print("URL:", obj.properties["url"] or obj.properties["hn_url"])
    print("-" * 100)

Distance: 0.26538312435150146
Title: Show HN: GreenKube – Open-source K8s cost and CO2 optimization engine
Query: fastapi
Points: 2
Comments: 0
URL: https://github.com/GreenKubeCloud/GreenKube
----------------------------------------------------------------------------------------------------
Distance: 0.2851896286010742
Title: Ask HN: What is your (AI) dev tech stack / workflow?
Query: fastapi
Points: 143
Comments: 124
URL: https://news.ycombinator.com/item?id=48413629
----------------------------------------------------------------------------------------------------
Distance: 0.28547096252441406
Title: Show HN: Microcodegen.py – PRD → FastAPI app, one file, no LLM calls
Query: fastapi
Points: 3
Comments: 0
URL: https://github.com/Anioko/microcodegen
----------------------------------------------------------------------------------------------------
Distance: 0.29968535900115967
Title: Cursorfy – recursive Cursor SDK and FastAPI dashboard that sees itself
Query: fastapi
Points: 2
Com

In [17]:
filters = (
    Filter.by_property("points").greater_or_equal(10)
    & Filter.by_property("comments_count").greater_or_equal(2)
)

response = hn_stories.query.near_text(
    query="interesting technical discussion about AI, backend engineering or infrastructure",
    target_vector="story_vector",
    filters=filters,
    limit=8,
    return_properties=[
        "title",
        "query",
        "points",
        "comments_count",
        "url",
        "hn_url",
        "summary",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Query:", obj.properties["query"])
    print("Points:", obj.properties["points"])
    print("Comments:", obj.properties["comments_count"])
    print("URL:", obj.properties["url"] or obj.properties["hn_url"])
    print("-" * 100)

Distance: 0.3007485866546631
Title: Ask HN: What is your (AI) dev tech stack / workflow?
Query: fastapi
Points: 143
Comments: 124
URL: https://news.ycombinator.com/item?id=48413629
----------------------------------------------------------------------------------------------------
Distance: 0.3009531497955322
Title: I Replaced My AI Agent's Flat Fact Store with a Graph Database
Query: vector database
Points: 10
Comments: 5
URL: https://news.ycombinator.com/item?id=48383578
----------------------------------------------------------------------------------------------------
Distance: 0.30697256326675415
Title: Show HN: Lowfat – pluggable CLI filter that saved 91.8% of my LLM tokens
Query: docker
Points: 138
Comments: 70
URL: https://github.com/zdk/lowfat
----------------------------------------------------------------------------------------------------
Distance: 0.3225480914115906
Title: Show HN: Cost.dev (YC W21) – making agents cost-aware and cheaper to call
Query: developer tools
Poi

In [18]:
response = hn_stories.generate.near_text(
    query="What are the most interesting current developer technology topics?",
    target_vector="story_vector",
    grouped_task=(
        "Act as a concise technology analyst. "
        "Use only the retrieved Hacker News story records. "
        "Create a short briefing about the main developer technology topics. "
        "Mention relevant titles, points, comment counts, and why they may be interesting. "
        "Do not invent facts outside the retrieved records."
    ),
    limit=8,
    return_properties=[
        "title",
        "query",
        "points",
        "comments_count",
        "url",
        "hn_url",
        "summary",
    ],
)

print(response.generated)

print("\nSources:")
for obj in response.objects:
    print(
        "-",
        obj.properties["title"],
        "| points:",
        obj.properties["points"],
        "| comments:",
        obj.properties["comments_count"],
    )

The main developer technology topics center heavily on **Artificial Intelligence (AI) and its practical application in development**.

**Key Topics:**

1.  **AI Development Stacks and Workflows:**

Sources:
- A curated list of AI for developers | points: 4 | comments: 0
- Show HN: On-device transcriber that's 97% accurate at identifying speakers | points: 22 | comments: 6
- Ask HN: What is your (AI) dev tech stack / workflow? | points: 143 | comments: 124
- Show HN: Lich, start a dev stack per coding agent in parallel | points: 6 | comments: 2
- Show HN: Cost.dev (YC W21) – making agents cost-aware and cheaper to call | points: 35 | comments: 22
- France, Germany, and the Netherlands public admin co-developed public tools | points: 4 | comments: 0
- Gemma 4 QAT models: Optimizing compression for mobile and laptop efficiency | points: 376 | comments: 115
- Show HN: Agents Remember – Git-aware memory for coding agents | points: 3 | comments: 2


In [19]:
def tech_briefing(
    user_interest: str,
    *,
    min_points: int | None = None,
    min_comments: int | None = None,
    limit: int = 8,
) -> None:
    filters = None

    if min_points is not None:
        points_filter = Filter.by_property("points").greater_or_equal(min_points)
        filters = points_filter if filters is None else filters & points_filter

    if min_comments is not None:
        comments_filter = Filter.by_property("comments_count").greater_or_equal(
            min_comments
        )
        filters = comments_filter if filters is None else filters & comments_filter

    response = hn_stories.generate.near_text(
        query=user_interest,
        target_vector="story_vector",
        filters=filters,
        grouped_task=(
            "Use only the retrieved Hacker News story records. "
            "Prepare a concise technical briefing for a Python/backend developer. "
            "Group stories by theme where possible. "
            "Mention story title, points and comment count. "
            "Do not invent missing details."
        ),
        limit=limit,
        return_properties=[
            "title",
            "query",
            "points",
            "comments_count",
            "url",
            "hn_url",
            "summary",
        ],
    )

    print("USER INTEREST:")
    print(user_interest)
    print("MIN POINTS:", min_points)
    print("MIN COMMENTS:", min_comments)
    print("-" * 100)

    print(response.generated)

    print("\nRecords used:")
    for obj in response.objects:
        print(
            "-",
            obj.properties["title"],
            "| query:",
            obj.properties["query"],
            "| points:",
            obj.properties["points"],
            "| comments:",
            obj.properties["comments_count"],
            "| url:",
            obj.properties["url"] or obj.properties["hn_url"],
        )

In [22]:
tech_briefing(
    "AI tools for developers, LLM applications, vector databases and RAG",
    min_points=5,
    min_comments=1,
)

USER INTEREST:
AI tools for developers, LLM applications, vector databases and RAG
MIN POINTS: 5
MIN COMMENTS: 1
----------------------------------------------------------------------------------------------------
Here's a concise technical briefing for Python/backend developers, based on recent Hacker News stories:

---

**Technical Briefing: AI/LLM Trends & Developer Tools**

Recent Hacker News discussions highlight a strong focus on AI/LLM development, optimization, and general developer tooling.

**1. AI/LLM Development & Optimization**
Several stories address the practicalities and performance of AI and Large Language Models:
*   **Ask HN: What is your (AI) dev tech stack / workflow?** (143 points, 124 comments) – A popular discussion on current AI development practices.
*   **Gemma 4 QAT models: Optimizing compression for mobile and laptop efficiency** (376 points, 115 comments) –

Records used:
- I Replaced My AI Agent's Flat Fact Store with a Graph Database | query: vector data

In [23]:
tech_briefing(
    "Python backend engineering with FastAPI, PostgreSQL, Docker and Kubernetes",
    min_points=5,
)

USER INTEREST:
Python backend engineering with FastAPI, PostgreSQL, Docker and Kubernetes
MIN POINTS: 5
MIN COMMENTS: None
----------------------------------------------------------------------------------------------------
Here's a concise technical briefing for Python/backend developers based on recent Hacker News stories:

### Technical Briefing: Recent Hacker News Trends for Python/Backend Developers

This briefing highlights recent discussions and tools relevant to Python and backend development, grouped by theme.

#### AI/LLM

Records used:
- Ask HN: What is your (AI) dev tech stack / workflow? | query: fastapi | points: 143 | comments: 124 | url: https://news.ycombinator.com/item?id=48413629
- Show HN: Mercek – A Desktop IDE for AWS ECS | query: kubernetes | points: 62 | comments: 29 | url: https://www.mercek.dev/
- Show HN: I embedded 685M public texts in 32 minutes (on 8x A100, Rust, TensorRT) | query: docker | points: 7 | comments: 0 | url: https://github.com/Artain-AI/ignite

In [24]:
tech_briefing(
    "open source developer tools and infrastructure",
    min_points=10,
)

USER INTEREST:
open source developer tools and infrastructure
MIN POINTS: 10
MIN COMMENTS: None
----------------------------------------------------------------------------------------------------
Here's a concise technical briefing based on the retrieved Hacker News stories:

---

**Technical Briefing for Python/Backend Developers**

This briefing summarizes recent Hacker News stories relevant to

Records used:
- Show HN: Cost.dev (YC W21) – making agents cost-aware and cheaper to call | query: developer tools | points: 35 | comments: 22 | url: https://cost.dev/
- Gemma 4 QAT models: Optimizing compression for mobile and laptop efficiency | query: developer tools | points: 376 | comments: 115 | url: https://blog.google/innovation-and-ai/technology/developers-tools/quantization-aware-training-gemma-4/
- Show HN: On-device transcriber that's 97% accurate at identifying speakers | query: developer tools | points: 22 | comments: 6 | url: https://mimicscribe.app/
- Show HN: Lowfat – plugga

In [25]:
from google import genai

google_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

for model in google_client.models.list():
    model_name = model.name
    if "embedding" in model_name.lower():
        print(model_name)

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [26]:
from google import genai

google_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

try:
    result = google_client.models.embed_content(
        model="gemini-embedding-2",
        contents="task: search result | query: Python backend with FastAPI and Docker",
    )

    embedding = result.embeddings[0].values

    print("gemini-embedding-2 works")
    print("Embedding length:", len(embedding))
    print("First values:", embedding[:5])

except Exception as error:
    print("gemini-embedding-2 failed")
    print(type(error).__name__)
    print(error)

gemini-embedding-2 works
Embedding length: 3072
First values: [-0.012006017, 0.0015956052, 0.013978046, -0.00071168103, -0.018637389]
